In [50]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf,month,sum ,count

In [51]:
spark = SparkSession.builder.appName("Case-study-app").getOrCreate()

In [52]:
customers = spark.read.option("header",True).csv("data/customers.csv")
products =  spark.read.option("header",True).csv("data/products.csv")
orders = spark.read.option("header",True).csv("data/orders.csv")
order_items = spark.read.option("header",True).csv("data/order_items.csv")
returns = spark.read.option("header",True).csv("data/returns.csv")

In [53]:
customers.count()

100000

In [54]:
orders.count()

1000000

In [55]:
returns.count()

100000

In [56]:
order_items.count()

3000000

In [57]:
products.count()

50000

In [58]:
products.createOrReplaceTempView("Product_table")
customers.createOrReplaceTempView("Customer_table")
orders.createOrReplaceTempView("Order_table")
order_items.createOrReplaceTempView("Order_items_table")
returns.createOrReplaceTempView("Returns_table")

In [15]:
result = spark.sql('''
select category,SUM(unit_cost) AS total_sales
from Product_table 
group by category''')
result.show()

+--------------+------------------+
|      category|       total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [16]:
result.write.mode("overwrite").csv("output/q2",header=True)

In [17]:
customers.show(10)

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
|          6|   Customer_6| Detroit|   TX|       2022-08-17|        Standard|
|          7|   Customer_7| Houston|   NC|       2022-05-23|         Premium|
|          8|   Customer_8|Columbus|   NY|       2022-04-15|         Premium|
|          9|   Customer_9| Atlanta|   NY|       2023-11-24|             VIP|
|         10|  Customer_10| Chicago|   GA|       2023-07-13|    

In [18]:
products.show(10)

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electronics|Brand_A|   332.22|
|         3|   Product_3|        Sports|Brand_D|    68.58|
|         4|   Product_4|      Clothing|Brand_A|   729.19|
|         5|   Product_5|Home & Kitchen|Brand_D|   326.77|
|         6|   Product_6|        Beauty|Brand_E|   684.21|
|         7|   Product_7|          Toys|Brand_D|   634.58|
|         8|   Product_8|Home & Kitchen|Brand_B|   158.47|
|         9|   Product_9|   Electronics|Brand_C|    287.9|
|        10|  Product_10|      Clothing|Brand_A|    93.38|
+----------+------------+--------------+-------+---------+
only showing top 10 rows



In [19]:
orders.show(10)

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|payment_mode|order_status|
+--------+-----------+----------+------------+------------+
|       1|      54630|2024-01-25| Credit Card|   Delivered|
|       2|      22415|2024-07-08| Credit Card|   Delivered|
|       3|      20909|2024-01-23|         UPI|   Delivered|
|       4|      98027|2024-03-02| Credit Card|   Cancelled|
|       5|      31332|2024-12-17|         UPI|   Cancelled|
|       6|      61229|2024-02-13| Net Banking|   Cancelled|
|       7|      23352|2024-07-15|         UPI|   Delivered|
|       8|      63016|2024-01-12|  Debit Card|   Delivered|
|       9|      71151|2024-01-12|  Debit Card|    Returned|
|      10|      72658|2024-03-12| Credit Card|   Delivered|
+--------+-----------+----------+------------+------------+
only showing top 10 rows



In [20]:
order_items.show(10)

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|       2|       203.02|
|            3|  426242|      2818|       5|      1061.81|
|            4|  236965|     35931|       5|      1005.49|
|            5|  289552|     48310|       4|       540.78|
|            6|  631355|     29023|       4|        663.7|
|            7|  813168|      6493|       2|       761.51|
|            8|  711584|     12240|       3|       146.62|
|            9|  485160|     15636|       4|      1033.48|
|           10|   76601|     40856|       3|        887.2|
+-------------+--------+----------+--------+-------------+
only showing top 10 rows



In [32]:
returns.show(10)

+---------+--------+-----------+------------------+
|return_id|order_id|return_date|     return_reason|
+---------+--------+-----------+------------------+
|        1|  391349| 2024-08-06|      Changed Mind|
|        2|  456171| 2024-06-28|   Arrived Damaged|
|        3|  977675| 2024-09-26|        Size Issue|
|        4|   80452| 2024-07-30|   Arrived Damaged|
|        5|   10920| 2024-06-15|        Size Issue|
|        6|  365396| 2024-08-16|Wrong Item Shipped|
|        7|  675259| 2024-10-02|   Arrived Damaged|
|        8|  871061| 2024-11-02| Defective Product|
|        9|  996307| 2024-04-11| Defective Product|
|       10|  390334| 2024-07-06|Wrong Item Shipped|
+---------+--------+-----------+------------------+
only showing top 10 rows



In [22]:
result2 = spark.sql('''
select customer_id ,sum(oi.quantity*oi.selling_price) AS Total_amount
from Order_table o
join Order_items_table oi
on o.order_id = oi.order_id
group by customer_id
order by Total_amount desc
limit 10''')
result2.show(10)

[Stage 65:>                                                         (0 + 2) / 2]

+-----------+------------------+
|customer_id|      Total_amount|
+-----------+------------------+
|      93094|181569.68000000005|
|      64560|169060.39999999997|
|      23289|          161573.8|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      52034|         152680.05|
|      40442|151037.32000000004|
|      60528|         148691.95|
|      84830|         148363.84|
|      82593|         148281.04|
+-----------+------------------+



In [23]:
result2.write.mode("overwrite").csv("output/intermediate",header=True)

In [24]:
intermediate = spark.read.option("header",True).csv("output/intermediate")
intermediate.createOrReplaceTempView("result2")

In [25]:
result3 = spark.sql('''
select customer_name, Total_amount
from result2 r
join Customer_table c
on r.customer_id = c.customer_id''')
result3.show()

+--------------+------------------+
| customer_name|      Total_amount|
+--------------+------------------+
|Customer_23289|          161573.8|
|Customer_40442|151037.32000000004|
|Customer_52034|         152680.05|
|Customer_52275|153364.78999999998|
|Customer_60528|         148691.95|
|Customer_61218|         153067.55|
|Customer_64560|169060.39999999997|
|Customer_82593|         148281.04|
|Customer_84830|         148363.84|
|Customer_93094|181569.68000000005|
+--------------+------------------+



In [26]:
result3.write.mode("overwrite").csv("output/q3",header=True)

In [29]:
# question 4
result4 = (
    orders
    .join(order_items,"order_id") 
    .groupBy(month("order_date").alias("Month"))
    .agg(
        sum(
            col("selling_price")*col("quantity")
        ).alias("Total_amount")
    )
    .orderBy("Month")
)
result4.show()
            

[Stage 90:>                                                         (0 + 2) / 2]

+-----+--------------------+
|Month|        Total_amount|
+-----+--------------------+
|    1| 4.445777757600014E8|
|    2|4.1536614419999766E8|
|    3| 4.436282454099968E8|
|    4|4.2782097433999556E8|
|    5|4.4481061894999766E8|
|    6| 4.317051540600035E8|
|    7| 4.436705191200028E8|
|    8| 4.410951770200006E8|
|    9|4.3107152608000004E8|
|   10| 4.413637893100021E8|
|   11|4.3362336404000014E8|
|   12| 4.427129083499984E8|
+-----+--------------------+



In [68]:
result4.write.mode("overwrite").csv("output/q4",header=True)

In [35]:
# question 5
total_order=order_items.join(products, on="product_id").groupBy("category").agg(count("quantity").alias("Total_orders"))

In [36]:
total_order.show()

[Stage 103:>                                                        (0 + 2) / 2]

+--------------+------------+
|      category|Total_orders|
+--------------+------------+
|Home & Kitchen|      434034|
|        Sports|      424412|
|   Electronics|      425896|
|      Clothing|      427607|
|         Books|      427086|
|        Beauty|      430547|
|          Toys|      430418|
+--------------+------------+



In [37]:
total_order.write.mode("overwrite").csv("temp/total",header=True)

In [39]:
total_order =  spark.read.option("header",True).csv("temp/total/t1d.csv")
total_order.createOrReplaceTempView("total_order")

In [42]:
order_category_qty= spark.sql('''
    select oi.order_id, oi.quantity, p.category
    from Order_items_table oi join Product_table p
    on oi.product_id=p.product_id
    '''
)

In [43]:
order_category_qty.show()

+--------+--------+--------------+
|order_id|quantity|      category|
+--------+--------+--------------+
|  227444|       5|          Toys|
|   32708|       2|   Electronics|
|  426242|       5|        Beauty|
|  236965|       5|          Toys|
|  289552|       4|      Clothing|
|  631355|       4|      Clothing|
|  813168|       2|        Beauty|
|  711584|       3|        Beauty|
|  485160|       4|         Books|
|   76601|       3|Home & Kitchen|
|  531530|       2|      Clothing|
|  366262|       4|      Clothing|
|  198684|       1|         Books|
|   87222|       2|        Beauty|
|  220287|       3|Home & Kitchen|
|  567551|       5|Home & Kitchen|
|  950074|       2|      Clothing|
|  452560|       2|         Books|
|  765970|       2|      Clothing|
|  456353|       3|         Books|
+--------+--------+--------------+
only showing top 20 rows



In [44]:
order_category_qty.write.mode("overwrite").csv("temp/order_category_qty",header=True)

In [45]:
ocq =  spark.read.option("header",True).csv("temp/order_category_qty/t1b.csv")

In [46]:
ocq.createOrReplaceTempView("ocq")

In [59]:
ret_qc=spark.sql('''
select count (t.quantity) as Quantity, t.category
from Returns_table r join ocq t
on t.order_id=r.order_id
group by t.category
''')

In [60]:
ret_qc.write.mode("overwrite").csv("temp/ret_qc",header=True)

In [61]:
ret_qc =  spark.read.option("header",True).csv("temp/ret_qc/t1c.csv")

In [62]:
ret_qc.createOrReplaceTempView("ret_qc")

In [65]:
ret_qc.show()

+--------+--------------+
|Quantity|      category|
+--------+--------------+
|   20239|Home & Kitchen|
|   19929|        Sports|
|   20169|   Electronics|
|   20035|      Clothing|
|   20059|         Books|
|   20198|        Beauty|
|   20362|          Toys|
+--------+--------------+



In [66]:
final=spark.sql('''
select r.category, ((r.Quantity/t.Total_orders)*100) as percentage
from ret_qc r join total_order t
on r.category = t.category
''')

In [67]:
final.show()

+--------------+------------------+
|      category|        percentage|
+--------------+------------------+
|Home & Kitchen| 4.662998751249902|
|        Sports| 4.695673072391921|
|   Electronics| 4.735663166594661|
|      Clothing| 4.685376993360726|
|         Books| 4.696712137602263|
|        Beauty|4.6912416066074085|
|          Toys| 4.730750108034516|
+--------------+------------------+



In [69]:
final.write.mode("overwrite").csv("output/q5",header=True)

In [71]:
# question 6
q6=spark.sql(
    '''
    Select count(o.customer_id) as Total,c.state, o.payment_mode
    from Customer_table c join Order_table o
    on o.customer_id=c.customer_id
    group by c.state,o.payment_mode
    '''
)

In [72]:
q6.show()

[Stage 165:>                                                        (0 + 2) / 2]

+-----+-----+----------------+
|Total|state|    payment_mode|
+-----+-----+----------------+
|19629|   FL|     Net Banking|
|19596|   NC|     Net Banking|
|19733|   GA|      Debit Card|
|19930|   OH|             UPI|
|20205|   MI|Cash on Delivery|
|20498|   IL|Cash on Delivery|
|20351|   OH|     Net Banking|
|19874|   TX|     Credit Card|
|20359|   IL|             UPI|
|20108|   NY|             UPI|
|20229|   NY|     Net Banking|
|20065|   TX|             UPI|
|19722|   GA|     Credit Card|
|19988|   TX|      Debit Card|
|20116|   MI|     Net Banking|
|20208|   NY|Cash on Delivery|
|20024|   CA|      Debit Card|
|20404|   IL|     Net Banking|
|19979|   CA|     Credit Card|
|19794|   OH|     Credit Card|
+-----+-----+----------------+
only showing top 20 rows



In [73]:
q6.createOrReplaceTempView("payment_counts")

In [74]:
max_count = spark.sql("""
SELECT
    state,
    MAX(Total) AS max_orders
FROM payment_counts
GROUP BY state
""")

max_count.createOrReplaceTempView("max_count")

In [75]:
max_count.show()

[Stage 169:============================>                            (1 + 1) / 2]

+-----+----------+
|state|max_orders|
+-----+----------+
|   MI|     20416|
|   CA|     20246|
|   NC|     19596|
|   IL|     20498|
|   WA|     20244|
|   OH|     20351|
|   NY|     20369|
|   TX|     20065|
|   GA|     20041|
|   FL|     20010|
+-----+----------+



In [77]:
result6= spark.sql('''
SELECT
    p.state,
    p.payment_mode,
    p.Total
FROM payment_counts p
JOIN max_count m
ON p.state = m.state
AND p.Total = m.max_orders
''')

In [78]:
result6.show()

[Stage 176:>                                                        (0 + 2) / 2]

+-----+----------------+-----+
|state|    payment_mode|Total|
+-----+----------------+-----+
|   NC|     Net Banking|19596|
|   IL|Cash on Delivery|20498|
|   OH|     Net Banking|20351|
|   TX|             UPI|20065|
|   GA|     Net Banking|20041|
|   WA|             UPI|20244|
|   CA|             UPI|20246|
|   FL|      Debit Card|20010|
|   NY|      Debit Card|20369|
|   MI|      Debit Card|20416|
+-----+----------------+-----+



In [79]:
result6.write.mode("overwrite").csv("output/q6",header=True)